# Day 2 Reference Solution: Orders, inventory & delivery (ORD)

**JUDGES ONLY. Never share with teams.**

This is the full registry the ORD starter is asking teams to build toward: **four Kind 1 patterns** (Order, Customer, Driver, Return) and **four Kind 2 candidates deliberately rejected** (Dock, Zone, Quantity, Season), each with a reason.

It runs the public pack *and* the hidden pack. Because the starter's `grounded_answer` is fixed (record IDs in the context, the checked verdict beside each pair, `temperature=0`), the answer follows the graph verdict whenever retrieval returns the two records, so the **registry is what is being judged**.

The offline checker (`day2_offline_check.py --dataset ORD`) reproduces the structural verdicts without Neo4j or an LLM.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [ ]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"
LLM_BASE_URL = None  # None = OpenAI. Otherwise your provider's OpenAI-compatible URL

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

### Connect to Neo4j Aura

Create a free [Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance if you haven't, then add `NEO4J_URI`, `NEO4J_USER` and `NEO4J_PASSWORD` to Colab's Secrets manager (padlock icon). If you hit `ServiceUnavailable`, the instance auto-paused: re-run this cell.

**Cannot use Neo4j?** Skip this cell. Replace the graph-building cell (section 7) and `graph_check` (section 8) with lookup-table versions that keep the same return shape. The Student Setup Guide shows the table structure.

In [ ]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

print(run_read("RETURN 'Connection successful' AS message")[0]["message"])

## 2. The dataset: campus operations & student services

179 records in 14 categories. Four pairs of categories deliberately share vocabulary (money, rooms, passes/permits, events). Most records stand alone; some name a person, a club or a ticket.

In [ ]:
DATASET_ID = "ORD"
DATASET_NAME = "Orders, inventory & delivery"

rec_categories = {
    'Order_Intake': [
        'A new online order was received and queued for payment verification',
        'Order validation flagged a mismatched shipping postcode',
        'A bulk order from a corporate buyer was split into smaller batches',
        'An order was placed on hold pending address confirmation',
        'Customer cancelled an order before it was picked',
        'A pre-order for an upcoming product was accepted into the queue',
        'Duplicate order submission was detected and merged',
        'Gift order notes were attached to the packing instruction',
        'An express order was prioritised ahead of the standard queue',
        'Order O1042 was placed for a set of kitchen shelves and queued for payment verification',
        'Order O1063 was placed on hold pending address confirmation',
        'Order O1077 was partially fulfilled and the remaining items were backordered',
        'Order O1090 was split into two parcels because the items shipped from different warehouses',
    ],
    'Inventory_Counts': [
        'Cycle count found a small variance between system stock and shelf stock',
        'Safety stock thresholds were reviewed for fast-moving items',
        'A damaged pallet was written off from available inventory',
        'Stock transfer between two warehouses was recorded in the ledger',
        'Slow-moving items were flagged for a clearance discount',
        'Inventory snapshot was frozen for the quarterly audit',
        'A mis-scanned item was corrected in the stock system',
        'Reorder points were recalculated after a demand spike',
        'Obsolete stock was moved to the disposal list',
        'Order O1077 triggered a stock shortage alert for a backordered item',
        'Order O1077 reserved 500 units of printer paper from the main warehouse',
        'A stock transfer of 500 units of cleaning supplies was recorded between two warehouses',
        'A write-off of 240 units of expired snacks was approved after the audit',
        'A cycle count in Zone B found one pallet missing',
        'Return RMA-4471 was inspected and the blender was written off as scrap',
    ],
    'Warehouse_Storage': [
        'Pallet racking in the cold storage area was inspected for damage',
        'Slotting was optimised to place popular items near the packing stations',
        'A forklift battery was swapped at the charging bay',
        'Overflow stock was moved to the temporary storage yard',
        'Fire safety inspection was completed for the main storage hall',
        'Shelving labels were reprinted after a re-slotting exercise',
        'Temperature log for the cold storage was reviewed by the supervisor',
        'A mezzanine floor was cleared for seasonal storage',
        'Pest control treatment was scheduled for the dry goods section',
        'Zone B was reorganised to hold fast-moving products',
        'A forklift collision at Dock 4 was reported to the safety officer',
        'Driver D15 reported a damaged ramp at Dock 9 to the warehouse supervisor',
    ],
    'Inbound_Receiving': [
        'A supplier truck arrived at the gate for unloading',
        'Received goods were checked against the advance shipping notice',
        'A short shipment was reported by the receiving team',
        'Damaged cartons were photographed and logged at receipt',
        'Putaway of received pallets was completed before shift end',
        'Receiving team quarantined a delivery with a missing label',
        'An unscheduled supplier delivery was turned away at the gate',
        'Inbound appointment slots were rebooked after a traffic delay',
        'Receiving staff counted a mixed pallet carton by carton',
        'Dock 4 was reserved for an incoming supplier truck this morning',
        'A shipment of 240 units of phone cases arrived from a supplier',
        'Driver D12 waited ninety minutes at Dock 9 for a free loading slot',
    ],
    'Outbound_Delivery': [
        'A delivery was attempted but the recipient was not at home',
        'Same-day delivery slots filled up before noon',
        'A parcel was rerouted after the courier hub flagged a wrong address',
        'Delivery confirmation photo was uploaded by the driver',
        'A late delivery was escalated to the regional delivery manager',
        'Out-for-delivery notification was sent to the customer',
        'Last-mile delivery to a remote area was rescheduled for the next day',
        'A signature-required parcel was left with a neighbour by mistake',
        'Delivery route was re-sequenced after a road closure',
        'Order O1042 was delivered to the customer and signed for at the door',
        'Order O1051 was marked out for delivery by the courier',
        'Driver D07 completed the morning route with every parcel delivered',
        'Driver D12 reported a blocked access road on the evening route',
        "Customer C219's parcel was rerouted to the new address by the courier",
    ],
    'Returns_RMA': [
        'A customer requested a return for a defective item',
        'Returned goods were inspected and restocked as new',
        'A return was rejected because the return window had passed',
        'Returns team issued a prepaid return label to a customer',
        'A refurbished item was graded and moved to the outlet stock',
        'Return pickup was scheduled with the courier',
        'A wrong-item return was approved without inspection',
        'Returns processing backlog was cleared after the weekend',
        'A return reason survey was added to the return form',
        'Return RMA-4471 was opened for a defective blender and a prepaid label was issued',
        'Return RMA-3092 was approved for a wrong-colour item',
        'Customer C233 returned a pair of shoes within the return window',
    ],
    'Billing_Refunds': [
        'A refund was issued to the original payment method',
        "Billing corrected a duplicate charge on a customer's card",
        'Refund processing time was extended due to a bank holiday',
        'A partial refund was approved for a late delivery',
        'Invoice was regenerated after a billing address change',
        'A chargeback notice was received from a card issuer',
        'Store credit was issued in place of a cash refund',
        "Billing reconciled the day's card settlements",
        'A promotional discount was reversed after an order edit',
        'Customer C204 received a partial refund for the damaged parcel',
        'Order O1088 was charged a re-delivery fee after two failed delivery attempts',
    ],
    'Supplier_Payments': [
        'Supplier invoice was approved for payment after three-way matching',
        'A payment run was scheduled for the end of the month',
        'Supplier requested early payment in exchange for a discount',
        'Procurement disputed a supplier invoice for incorrect pricing',
        'A supplier payment was held pending a quality complaint',
        'Bank details of a supplier were verified before the next payment',
        'Credit note from a supplier was applied against an open invoice',
        'Accounts payable aged balances were reviewed by finance',
        'Freight charges were added to the supplier invoice',
    ],
    'Customer_Support': [
        'A customer asked for an update on a delayed parcel',
        'Support agent resolved a login problem on the order tracking page',
        'A complaint about damaged packaging was escalated to quality',
        'Chat support handled a question about delivery charges',
        'A customer asked to change the delivery address after dispatch',
        'Support team sent an apology voucher for a missed delivery slot',
        'A customer enquired about the return policy for sale items',
        'Support tickets were tagged by topic for the weekly review',
        'A customer praised the quick delivery in a feedback form',
        'Customer C204 complained that a parcel arrived with damaged packaging',
        'Customer C219 asked to change the delivery address after dispatch',
        'Order O1063 was the subject of a customer request to change the delivery address',
    ],
    'Carrier_Management': [
        'Carrier performance scorecard was shared with the logistics team',
        'A new regional courier was onboarded for last-mile coverage',
        'Carrier rates were renegotiated for heavy parcels',
        'A courier contract renewal was sent for legal review',
        'Carrier pickup time was moved earlier for the evening dispatch',
        'Customs paperwork was prepared for an international shipment',
        'A carrier strike caused a backlog at the sorting hub',
        'Fuel surcharge changes were communicated by the carrier',
        'Carrier tracking feed was fixed after an integration error',
        'Driver D07 was commended by the regional manager for on-time performance',
        'Driver D15 completed the new-joiner safety briefing',
        'Carrier surcharges were announced for the holiday season',
    ],
    'Packaging': [
        'Packing station was restocked with bubble wrap and tape',
        'A new eco-friendly mailer bag was approved for light items',
        'Fragile items were double-boxed before dispatch',
        'Packing slips were reprinted after a printer jam',
        'Oversized items were wrapped in corrugated board',
        'Packers were trained on the new gift wrapping process',
        'A packing error was caught at the final scan',
        'Cartons of the wrong size were returned to the supplier',
        'Shrink wrap was used to secure a pallet for dispatch',
        'Order O1051 was double-boxed because it contained a glass lamp',
        'Extra temporary packers were hired for the holiday season',
    ],
    'Demand_Planning': [
        'Forecast was revised upward after a successful marketing campaign',
        'Planners reviewed the promotional calendar for the next quarter',
        'Demand for winter wear was projected to rise in the coming weeks',
        'Planning team agreed on a buffer stock for a new launch',
        'A forecast accuracy report was shared with the category managers',
        "Seasonal demand curves were updated with last year's data",
        'Planners flagged a supplier lead time increase',
        'A product was discontinued from the forecast model',
        'Weekly sales and operations meeting reviewed capacity constraints',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. The pattern registry (the answer key)

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Order',
        "regex": r"\bOrder\s+(O\d{4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Order O1042',
        "why": 'An order number names one order: two records naming O1042 are about the same order.',
    },
    {
        "name": 'Customer',
        "regex": r"\bCustomer\s+(C\d{3})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Customer C204',
        "why": 'A customer ID names one customer; the damage complaint and the partial refund for C204 are about the same person.',
    },
    {
        "name": 'Driver',
        "regex": r"\bDriver\s+(D\d{2})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Driver D07',
        "why": 'A driver ID names one driver; the morning route and the commendation for D07 are about the same driver.',
    },
    {
        "name": 'Return',
        "regex": r"\b(RMA-\d{3,4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'RMA-4471',
        "why": 'A return authorisation number names one return case from the label being issued to the item being inspected.',
    },
]

REJECTED_CANDIDATES = [
    {"candidate": 'Dock numbers (Dock 4, Dock 9)', "kind": 2, "why_rejected": 'A dock is a place reused by many unrelated trucks and drivers: Dock 9 appears with Driver D12 waiting and Driver D15 reporting a ramp, two unrelated events.'},
    {"candidate": 'Zone letters (Zone B)', "kind": 2, "why_rejected": 'A zone is a storage area; a reorganisation and a cycle count in Zone B are separate events.'},
    {"candidate": 'Quantities (240 units, 500 units)', "kind": 2, "why_rejected": 'The same quantity can appear in unrelated records by coincidence: 500 units of printer paper reserved vs 500 units of cleaning supplies transferred.'},
    {"candidate": 'Season (holiday season)', "kind": 2, "why_rejected": 'A season is a time period shared by many unrelated decisions, so linking on it would connect nearly everything.'},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 6. Build the graph

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

run_write("MATCH (n) DETACH DELETE n")

# One :Event node per record
event_rows = [{"id": f"R{i}", "text": t, "category": c}
              for i, (t, c) in enumerate(zip(rec_texts, rec_labels))]
run_write("""
    UNWIND $rows AS row
    CREATE (e:Event {id: row.id, text: row.text, category: row.category})
""", rows=event_rows)

# One entity node per distinct identifier; records naming it are linked through it
by_label = {}
for i, text in enumerate(rec_texts):
    for name, ident in extract_entities(text):
        by_label.setdefault(name, []).append({"eid": f"R{i}", "xid": ident})

for name, rows in by_label.items():       # name was validated: letters/digits/underscore only
    run_write(f"""
        UNWIND $rows AS r
        MATCH (e:Event {{id: r.eid}})
        MERGE (x:{name} {{id: r.xid}})
        MERGE (e)-[:MENTIONS]->(x)
    """, rows=rows)

print("Graph built from your registry:")
for name, rows in by_label.items():
    ids = {}
    for r in rows:
        ids.setdefault(r["xid"], 0)
        ids[r["xid"]] += 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {len(rows):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

## 7. Grounded answer

In [ ]:
def graph_check(id_a, id_b):
    """Asks Neo4j whether a path connects two records, and through which entities."""
    rows = run_read("""
        MATCH (a:Event {id: $a}), (b:Event {id: $b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN p IS NULL AS disconnected,
               [n IN nodes(p) WHERE NOT n:Event | labels(n)[0] + ' ' + n.id] AS via
    """, a=id_a, b=id_b)
    return rows[0] if rows else None


GROUNDED_PROMPT = """You answer questions about online retail fulfilment (orders, stock, warehouse, couriers, returns) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Quantities, dock numbers, zone letters, seasons, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 8. Question packs

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': "Was the refund issued to the customer's card paid out of the supplier payment run "
              'scheduled for month end?',
  'trap_records': ['A refund was issued to the original payment method',
                   'A payment run was scheduled for the end of the month'],
  'correct_answer_must_state': 'No stated connection: a customer refund and a supplier payment run '
                               'are separate records, both about money.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (money)'},
 {'id': 'P2',
  'question': 'Was the overflow stock moved to the temporary yard because the supplier truck '
              'arrived at the gate?',
  'trap_records': ['Overflow stock was moved to the temporary storage yard',
                   'A supplier truck arrived at the gate for unloading'],
  'correct_answer_must_state': 'No stated connection: a storage move and a truck arrival share no '
                               'order, shipment or named entity.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (place)'},
 {'id': 'P3',
  'question': 'Was the parcel rerouted because of the carrier strike at the sorting hub?',
  'trap_records': ['A parcel was rerouted after the courier hub flagged a wrong address',
                   'A carrier strike caused a backlog at the sorting hub'],
  'correct_answer_must_state': 'No stated connection: the reroute cites a wrong address, and '
                               'nothing ties it to the strike.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (movement)'},
 {'id': 'P4',
  'question': "Was the customer's defective-item return the same case as the damaged packaging "
              'complaint?',
  'trap_records': ['A customer requested a return for a defective item',
                   'A complaint about damaged packaging was escalated to quality'],
  'correct_answer_must_state': 'No evidence these two records name the same customer or order; '
                               'they are separate records.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (complaint)'},
 {'id': 'P5',
  'question': 'Did the incoming supplier truck cause the forklift collision at Dock 4?',
  'trap_records': ['Dock 4 was reserved for an incoming supplier truck this morning',
                   'A forklift collision at Dock 4 was reported to the safety officer'],
  'correct_answer_must_state': 'No stated connection: both mention Dock 4, but a dock is a place '
                               'reused by many trucks, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: dock'},
 {'id': 'P6',
  'question': 'Was the pallet missing in the cycle count lost during the Zone B reorganisation?',
  'trap_records': ['Zone B was reorganised to hold fast-moving products',
                   'A cycle count in Zone B found one pallet missing'],
  'correct_answer_must_state': 'No stated connection: sharing a zone name is not sharing an event.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: zone'},
 {'id': 'P7',
  'question': 'Were the 240 units of phone cases the same units that were written off?',
  'trap_records': ['A shipment of 240 units of phone cases arrived from a supplier',
                   'A write-off of 240 units of expired snacks was approved after the audit'],
  'correct_answer_must_state': 'No stated connection: the quantities match by coincidence, and the '
                               'products differ.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: quantity'},
 {'id': 'P8',
  'question': 'Was the forklift battery swapped because the forecast was revised upward?',
  'trap_records': ['A forklift battery was swapped at the charging bay',
                   'Forecast was revised upward after a successful marketing campaign'],
  'correct_answer_must_state': 'No stated connection between a battery swap and a forecast '
                               'revision; any causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Was Order O1051 double-boxed and also marked out for delivery?',
  'shared_identifier': 'Order O1051',
  'shared_records': ['Order O1051 was double-boxed because it contained a glass lamp',
                     'Order O1051 was marked out for delivery by the courier'],
  'correct_answer_must_state': 'Yes -- both records name Order O1051.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Was Order O1063 put on hold, and did a customer later ask to change its delivery '
              'address?',
  'shared_identifier': 'Order O1063',
  'shared_records': ['Order O1063 was placed on hold pending address confirmation',
                     'Order O1063 was the subject of a customer request to change the delivery '
                     'address'],
  'correct_answer_must_state': 'Yes -- both records name Order O1063.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Did Customer C204 both report damaged packaging and receive a partial refund?',
  'shared_identifier': 'Customer C204',
  'shared_records': ['Customer C204 complained that a parcel arrived with damaged packaging',
                     'Customer C204 received a partial refund for the damaged parcel'],
  'correct_answer_must_state': 'Yes -- both records name Customer C204.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
HIDDEN_ADVERSARIAL = [{'id': 'H1',
  'question': "Was the duplicate charge on the customer's card connected to the supplier invoice "
              'approved after three-way matching?',
  'trap_records': ["Billing corrected a duplicate charge on a customer's card",
                   'Supplier invoice was approved for payment after three-way matching'],
  'correct_answer_must_state': 'No stated connection: a card correction and a supplier invoice '
                               'approval are independent records.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H2',
  'question': 'Did the receiving check against the shipping notice cause the variance found in the '
              'cycle count?',
  'trap_records': ['Received goods were checked against the advance shipping notice',
                   'Cycle count found a small variance between system stock and shelf stock'],
  'correct_answer_must_state': 'No stated connection: nothing ties the receiving check to the '
                               'stock variance.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H3',
  'question': 'Was the return rejected because the earlier delivery attempt failed?',
  'trap_records': ['A delivery was attempted but the recipient was not at home',
                   'A return was rejected because the return window had passed'],
  'correct_answer_must_state': 'No stated connection: the return was rejected for the lapsed '
                               'window, and no order or customer links it to the failed delivery.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H4',
  'question': 'Did onboarding the new regional courier cause the same-day delivery slots to fill '
              'up before noon?',
  'trap_records': ['A new regional courier was onboarded for last-mile coverage',
                   'Same-day delivery slots filled up before noon'],
  'correct_answer_must_state': 'No stated connection: both concern deliveries but name no shared '
                               'order, carrier or driver.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H5',
  'question': 'Was the supplier lead time increase flagged because of the pre-order accepted into '
              'the queue?',
  'trap_records': ['A pre-order for an upcoming product was accepted into the queue',
                   'Planners flagged a supplier lead time increase'],
  'correct_answer_must_state': 'No stated connection between a pre-order and a lead-time flag.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H6',
  'question': "Did the fire safety inspection affect the customer's question about the return "
              'policy?',
  'trap_records': ['Fire safety inspection was completed for the main storage hall',
                   'A customer enquired about the return policy for sale items'],
  'correct_answer_must_state': 'No connection: a facilities inspection and a policy enquiry do not '
                               'even share vocabulary; any causal story is fabricated.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H7',
  'question': 'Was the damaged ramp reported by Driver D15 at Dock 9 what delayed Driver D12 '
              'there?',
  'trap_records': ['Driver D15 reported a damaged ramp at Dock 9 to the warehouse supervisor',
                   'Driver D12 waited ninety minutes at Dock 9 for a free loading slot'],
  'correct_answer_must_state': 'No -- two different drivers (D12, D15) each mention Dock 9, but a '
                               'dock is a place reused over time, not an event identifier. Linking '
                               'by dock alone would wrongly connect them.',
  'expect_connected': False,
  'kind2_trap': 'Dock number (Dock 9)'},
 {'id': 'H8',
  'question': 'Was the stock transfer of 500 units made to fulfil the reservation on Order O1077?',
  'trap_records': ['Order O1077 reserved 500 units of printer paper from the main warehouse',
                   'A stock transfer of 500 units of cleaning supplies was recorded between two '
                   'warehouses'],
  'correct_answer_must_state': "Order O1077's 500-unit reservation is recorded, and a 500-unit "
                               'transfer exists, but the transfer names no order and a different '
                               'product. The matching quantity is incidental, not an event '
                               'identifier.',
  'expect_connected': False,
  'kind2_trap': 'Quantity (500 units)'},
 {'id': 'H9',
  'question': 'Were the temporary packers hired because of the carrier surcharges announced for '
              'the holiday season?',
  'trap_records': ['Extra temporary packers were hired for the holiday season',
                   'Carrier surcharges were announced for the holiday season'],
  'correct_answer_must_state': 'No stated connection: both mention the holiday season, a time '
                               'period shared by many unrelated decisions.',
  'expect_connected': False,
  'kind2_trap': 'Season (holiday season)'}]

HIDDEN_POSITIVE = [{'id': 'HP1',
  'question': 'Was the defective blender return RMA-4471 inspected after the prepaid label was '
              'issued?',
  'shared_identifier': 'RMA-4471',
  'shared_records': ['Return RMA-4471 was opened for a defective blender and a prepaid label was '
                     'issued',
                     'Return RMA-4471 was inspected and the blender was written off as scrap'],
  'correct_answer_must_state': 'Yes -- both records carry return RMA-4471.',
  'expect_connected': True},
 {'id': 'HP2',
  'question': 'Was the kitchen shelves order O1042 delivered after payment verification?',
  'shared_identifier': 'Order O1042',
  'shared_records': ['Order O1042 was placed for a set of kitchen shelves and queued for payment '
                     'verification',
                     'Order O1042 was delivered to the customer and signed for at the door'],
  'correct_answer_must_state': 'Yes -- both records name Order O1042.',
  'expect_connected': True},
 {'id': 'HP3',
  'question': 'Did Driver D07 complete the morning route, and was the same driver commended for '
              'on-time performance?',
  'shared_identifier': 'Driver D07',
  'shared_records': ['Driver D07 completed the morning route with every parcel delivered',
                     'Driver D07 was commended by the regional manager for on-time performance'],
  'correct_answer_must_state': 'Yes -- both records name Driver D07.',
  'expect_connected': True},
 {'id': 'HP4',
  'question': 'Did Customer C219 ask to change the delivery address, and was the parcel then '
              'rerouted?',
  'shared_identifier': 'Customer C219',
  'shared_records': ['Customer C219 asked to change the delivery address after dispatch',
                     "Customer C219's parcel was rerouted to the new address by the courier"],
  'correct_answer_must_state': 'Yes -- both records name Customer C219.',
  'expect_connected': True}]

print(f"Hidden pack: {len(HIDDEN_ADVERSARIAL)} adversarial + {len(HIDDEN_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### 8a. Public pack

In [ ]:
pub_adv = run_pack(PUBLIC_ADVERSARIAL)
pub_pos = run_pack(PUBLIC_POSITIVE)

### 8b. HIDDEN pack (judges)

Expect 9/9 adversarial NOT connected and 4/4 positive CONNECTED at the graph level. Read the LLM answers as well: they should say "no recorded link" for adversarial questions, and affirm and name the shared entity for positives. Where baseline and grounded disagree is the story to tell.

In [ ]:
hid_adv = run_pack(HIDDEN_ADVERSARIAL)
hid_pos = run_pack(HIDDEN_POSITIVE)

## 9. Teaching points

1. **Kind 1 vs Kind 2.** Kind 2 traps: Dock 9 (H7), the 500-unit quantity (H8), the holiday season (H9); public pack has Dock 4, Zone B and 240 units. A team that adds those patterns fails exactly those questions.
2. **Coverage.** Hidden positives need Return (HP1), Driver (HP3) and Customer (HP4) on top of Order (HP2). A team that only adds the first obvious pattern fails the others.
3. **A baseline that fails only sometimes.** With a strict prompt the baseline can pass many of these questions; with the plain Week 2 prompt it is more likely to fabricate. Compare their answers on P5-P7 and H7-H9, and report what you actually see.
4. **Retrieval is separate.** If retrieval does not return both records, the graph check never sees them. Report that as a retrieval miss, not a grounding failure.